# ### Data yang digunakan adalah Healthcare Real-Time Risk Dataset yang bisa diakses di Kaggle melalui link ini (https://www.kaggle.com/datasets/alekhyaabburi/healthcare). Data ini mencakup 1000 pasien dengan 10 variabel fitur yang termasuk termasuk usia, jenis kelamin, BMI, status merokok, konsumsi alkohol, aktivitas fisik, durasi tidur, penyakit kronis, tingkat stres, dan tingkat risiko kesehatan. Variabel respon yang digunakan adalah tingkat stres dan variabel prediktor adalah konsumsi alkohol, aktivitas fisik, dan durasi tidur dalam analisis lanjutan dari pertemuan 9 dengan analisis Canonical Correlations.

## Langkah 0: Preprocessing Data

In [4]:
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

In [9]:
from google.colab import files
files.upload()

Saving df_healthcare.xlsx to df_healthcare.xlsx


{'df_healthcare.xlsx': b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x00NDF\xef\xa1\x01\x00\x00/\x07\x00\x00\x13\x00\x08\x02[Content_Types].xml \xa2\x04\x02(\xa0\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x0

In [10]:
df = pd.read_excel("df_healthcare.xlsx")
df

,Age,Gender,BMI,Smoking Status,Alcohol Consumption (per week),Physical Activity (hours/week),Sleep Duration (hours/day),Chronic Disease History,Stress Level (1-10),Health Risk Level
0,56,Other,2315,Never,1,1,49,Heart Disease,5,High
1,69,Male,2083,Former,3,98,72,NaN,8,Moderate
2,46,Female,2155,Never,0,97,54,Hypertension,7,Low
3,32,Other,2747,Former,18,40,48,Hypertension,8,Low
4,60,Male,2709,Current,9,73,59,Diabetes,8,High
...,...,...,...,...,...,...,...,...,...,...
995,18,Other,1908,Never,19,21,45,Diabetes,8,Moderate
996,35,Other,1686,Former,5,74,73,NaN,5,Moderate
997,49,Female,3136,Never,15,55,69,Heart Disease,8,High
998,64,Other,3446,Current,2,88,72,Diabetes,3,High


In [12]:
# Definisikan variabel X (prediktor) dan Y (respon)
X = df[['Alcohol Consumption (per week)', 'Physical Activity (hours/week)', 'Sleep Duration (hours/day)']]
Y = df[['Stress Level (1-10)']]

# Standarisasi Data
scaler_x = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_x.fit_transform(X)
Y_scaled = scaler_y.fit_transform(Y)

In [13]:
cca = CCA(n_components=1, scale=False, max_iter=1000)
U, V = cca.fit_transform(X_scaled, Y_scaled)

## Langkah 1: Tentukan Canonical Correlations

In [14]:
r = np.corrcoef(U[:, 0], V[:, 0])[0, 1]
print("Korelasi Kanonikal:", np.round(r, 3))
print("Nilai R-Square:", np.round(r**2, 3))

Korelasi Kanonikal: 0.079
Nilai R-Square: 0.006


## Langkah 2: Tentukan Loadings dan Cross Loadings

In [22]:
x_loadings = np.column_stack([
    [np.corrcoef(X_scaled[:, j],
                 U[:, i])[0, 1]
     for j in range(X_scaled.shape[1])]
    for i in range(cca.n_components)
])

y_loadings = np.column_stack([
    [np.corrcoef(Y_scaled[:, j],
                 V[:, i])[0, 1]
     for j in range(Y_scaled.shape[1])]
    for i in range(cca.n_components)
])


In [29]:
x_cross = np.column_stack([
    [np.corrcoef(X_scaled[:, j],
                 V[:, i])[0, 1]
     for j in range(X_scaled.shape[1])]
    for i in range(cca.n_components)
])

y_loadings = np.column_stack([
    [np.corrcoef(Y_scaled[:, j],
                 V[:, i])[0, 1]
     for j in range(Y_scaled.shape[1])]
    for i in range(cca.n_components)
])

# Langkah 3: Hitung Variance Extracted

In [30]:
ve_x = np.mean(x_loadings**2,
               axis=0)
ve_y = np.mean(y_loadings**2,
               axis=0)

print("Varians X:", ve_x)
print("Varians Y:", ve_y)

Varians X: [0.32765931]
Varians Y: [1.]


## Langkah 4: Hitung Indeks Redundancy

In [31]:
red_x = ve_x * (r**2)
red_y = ve_y * (r**2)

print("Redundancy X from Y:",
      np.round(red_x, 3))
print("Redundancy Y from X:",
      np.round(red_y, 3))

Redundancy X from Y: [0.002]
Redundancy Y from X: [0.006]


## Langkah 5: Hitung Wilks' Lambda dan Sequential Test

In [36]:
from scipy.stats import chi2

r_arr = np.atleast_1d(r)
n = X_scaled.shape[0]
p = X_scaled.shape[1]
q = Y_scaled.shape[1]
results = []

for k in range(len(r_arr)):
  wilks = np.prod(1 - r_arr[k:]**2)
  corr = n - 1 - (p + q + 1) / 2
  chi = -corr * np.log(wilks)
  df = (p - k) * (q - k)
  p_val = chi2.sf(chi, df)
  results.append([k + 1, wilks, chi, df, p_val])

test_table = pd.DataFrame(results, columns=[
    "Start_Function", "Wilks_Lambda",
    "Chi_Square", "df", "p_value"])
print(test_table.round(4))



   Start_Function  Wilks_Lambda  Chi_Square  df  p_value
0               1        0.9938      6.1856   3   0.1029


# Interpretasi Hasil
## Berdasarkan hasil di atas, diinterpretasikan bahwa:
### 1. Korelasi kanonikal sekitar 0,079, menunjukkan hubungan sangat lemah antara kelompok variabel prediktor dan respon.
### 2. Nilai R-Sqaured yang hanya 6% dari fungsi kanonikal variabel respon (tingkat stres) yang dapat dijelaskan oleh fungsi kanonikal variabel prediktor gaya hidup tersebut. Sisanya dipengaruhi oleh faktor lain di luar model.
### 3. Nilai Wilks' Lambda berada di sekitar nilai 0,9938, yaitu sangat mendekati 1, menandakan hampir tidak ada varians yang terikat antar kelompok variabel.
### 4. Variance Extracted (VE):
### X = 32,8% (Sedikit dirangkum keseluruhan variabel X)
### Y = 100% (Terpenuhi dirangkum keseluruhan variabel Y)
### 5. Indeks Redundansi: Indeks ini berada di sekitar 0,6%, menunjukkan kemampuan sangat rendah dalam menjelaskan variabilitas tingkat stres (Y) secara riil.
